In [3]:
# ==============================================================================
# PART 1: HYBRID FEATURE EXTRACTION (GPU REQUIRED)
# Switching to stable LeViT-256 to fix positional embedding mismatch
# ==============================================================================

import os
import time
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import numpy as np
from tqdm import tqdm

# --- Configuration (UPDATED) ---
DATA_ROOT = "/workspace/" 
TRAIN_PATH = os.path.join(DATA_ROOT, "Train")
VALID_PATH = os.path.join(DATA_ROOT, "Valid")
TEST_PATH = os.path.join(DATA_ROOT, "Test")

BATCH_SIZE = 64
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# *** CRITICAL CHANGE: SWITCH TO A STABLE IMAGE SIZE ***
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224 # <--- Changed from 384 to 256 for stable LeViT-256 weights 

# --- Data Transformation ---
necessary_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # Now 256x256
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

# --- 1. Load Datasets Directly from Separate Folders ---
print(f"Loading data from: {DATA_ROOT}")
train_set = datasets.ImageFolder(TRAIN_PATH, transform=necessary_transform)
val_set = datasets.ImageFolder(VALID_PATH, transform=necessary_transform)
test_set = datasets.ImageFolder(TEST_PATH, transform=necessary_transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"Train samples: {len(train_set)}, Valid samples: {len(val_set)}, Test samples: {len(test_set)}")


# --- 2. Hybrid Feature Extractor Model (ViT-Base Update) ---
class HybridFeatureExtractor(nn.Module):
    """Fuses features from ViT-Base and EfficientNetV2-S."""
    def __init__(self):
        super().__init__()
        
        # 1. ViT-Base branch (Vision Transformer) - NEW MODEL
        self.vit = timm.create_model(
            'vit_large_patch16_224', # <--- TARGET MODEL
            pretrained=True, 
            num_classes=0, 
            img_size=IMAGE_SIZE 
        )
        
        # 2. EfficientNetV2-S branch (CNN) - REMAINS THE SAME
        self.efficientnet = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0)

    def forward(self, x):
        vit_features = self.vit(x) # Use self.vit
        efficient_features = self.efficientnet(x)
        # Feature Fusion
        combined_features = torch.cat((vit_features, efficient_features), dim=1)
        return combined_features

# --- 3. Feature Extraction Function ---
def extract_features(data_loader, split_name):
    """Extracts features and records time."""
    start_time = time.time()
    model.eval()
    all_features = []
    all_labels = []
    
    print(f"Starting feature extraction for {split_name}...")
    with torch.no_grad():
        for images, labels in tqdm(data_loader):
            images = images.to(DEVICE)
            features = model(images).cpu().numpy()
            all_features.append(features)
            all_labels.extend(labels.tolist())

    features_matrix = np.concatenate(all_features, axis=0)
    labels_array = np.array(all_labels)
    elapsed_time = time.time() - start_time
    
    return features_matrix, labels_array, elapsed_time

# Instantiate Model
model = HybridFeatureExtractor().to(DEVICE)

# Run Extraction for all splits
X_train, y_train, train_time_feat = extract_features(train_loader, "Training")
X_val, y_val, val_time_feat = extract_features(val_loader, "Validation")
X_test, y_test, test_time_feat = extract_features(test_loader, "Testing")

print(f"\nFeature Extraction Times:")
print(f"Train Feat Time: {train_time_feat:.2f}s | Val Feat Time: {val_time_feat:.2f}s | Test Feat Time: {test_time_feat:.2f}s")

# Save Extracted Features
np.save(os.path.join(DATA_ROOT, 'X_train.npy'), X_train)
np.save(os.path.join(DATA_ROOT, 'y_train.npy'), y_train)
np.save(os.path.join(DATA_ROOT, 'X_valid.npy'), X_val)
np.save(os.path.join(DATA_ROOT, 'y_valid.npy'), y_val)
np.save(os.path.join(DATA_ROOT, 'X_test.npy'), X_test)
np.save(os.path.join(DATA_ROOT, 'y_test.npy'), y_test)

Using device: cuda
Loading data from: /workspace/
Train samples: 18898, Valid samples: 2362, Test samples: 2364


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Starting feature extraction for Training...


  0%|          | 0/296 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  1%|▏         | 4/296 [00:02<02:25,  2.01it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  3%|▎         | 10/296 [00:04<01:51,  2.56it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  9%|▉         | 26/296 [00:10<01:42,  2.63it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 296/296 [02:02<00:00,  2.42it/s]


Starting feature extraction for Validation...


  0%|          | 0/37 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  3%|▎         | 1/37 [00:01<00:43,  1.21s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 43%|████▎     | 16/37 [00:07<00:08,  2.48it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 37/37 [00:15<00:00,  2.33it/s]


Starting feature extraction for Testing...


  0%|          | 0/37 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  3%|▎         | 1/37 [00:01<00:49,  1.38s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 27%|██▋       | 10/37 [00:04<00:11,  2.41it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 37/37 [00:16<00:00,  2.31it/s]



Feature Extraction Times:
Train Feat Time: 122.31s | Val Feat Time: 15.89s | Test Feat Time: 16.07s


In [4]:
# ==============================================================================
# PART 2: XGBOOST TRAINING AND FULL METRICS (RE-RUN FOR ViT-BASE FEATURES)
# This entire block MUST be run again after new features are generated.
# ==============================================================================

import time
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    roc_auc_score, precision_recall_fscore_support, log_loss
)
import numpy as np
import pandas as pd
import os

# Define the feature times from your last successful feature extraction run (Part 1 for ViT-Base)
# NOTE: These values are placeholders; replace them with the actual output from your ViT-Base run if available.
# Since you didn't provide the times for the ViT-Base run, we'll assume they were similar to LeViT for now.
# However, you must ensure the variables X_train, etc. are defined by loading them.

DATA_ROOT = "/workspace/" 
X_train = np.load(os.path.join(DATA_ROOT, 'X_train.npy'))
y_train = np.load(os.path.join(DATA_ROOT, 'y_train.npy'))
X_val = np.load(os.path.join(DATA_ROOT, 'X_valid.npy'))
y_val = np.load(os.path.join(DATA_ROOT, 'y_valid.npy'))
X_test = np.load(os.path.join(DATA_ROOT, 'X_test.npy'))
y_test = np.load(os.path.join(DATA_ROOT, 'y_test.npy'))

print(f"Loaded training data shape: {X_train.shape}")
# Ensure you capture the new feature extraction times from the ViT-Base run.
# For simplicity in this response, we'll set placeholder times, but your environment should use the real ones.
train_time_feat = 65.0 
val_time_feat = 12.0
test_time_feat = 9.0


# --- 2. Train XGBoost Classifier and Time It (THIS IS THE CRITICAL RETRAINING STEP) ---
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='mlogloss',
    tree_method='hist', 
    random_state=42
)

print("Starting XGBoost training...")
start_time_train = time.time()
# XGBoost will now train on the new (18898, 2048) data shape
xgb_model.fit(X_train, y_train) 
end_time_train = time.time()
train_time_xgb = end_time_train - start_time_train


# --- 3. Define Metric Calculation Function (Using the Log Loss version) ---
def compute_metrics(X, y_true, split_name, model, is_train=False, feature_time=0):
    # ... (Keep the rest of the compute_metrics function as you last had it, 
    #      which includes log_loss calculation and timing logic) ...
    # Simplified placeholder for brevity:
    start_time_pred = time.time()
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)
    time_taken_pred = time.time() - start_time_pred
    
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    
    cm = confusion_matrix(y_true, y_pred)
    TP = np.diag(cm)
    FP = cm.sum(axis=0) - TP
    FN = cm.sum(axis=1) - TP
    TN = cm.sum() - (FP + FN + TP)
    epsilon = 1e-7
    TPR = np.mean(TP / (TP + FN + epsilon)) 
    FPR = np.mean(FP / (FP + TN + epsilon)) 
    
    try:
        auroc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        auroc = np.nan
    
    current_loss = 'N/A (XGBoost Objective)' 
    if not is_train:
        try:
            current_loss = log_loss(y_true, y_proba)
        except ValueError:
            current_loss = np.nan
        
    metrics = {
        f'{split_name} Accuracy': accuracy,
        f'{split_name} Precision': precision,
        f'{split_name} Recall': recall,
        f'{split_name} F1 Score': f1,
        f'{split_name} TPR (Macro Avg)': TPR,
        f'{split_name} FPR (Macro Avg)': FPR,
        f'{split_name} Loss': current_loss, 
        f'{split_name} Time (s)': time_taken_pred + feature_time,
    }
    
    if not is_train:
        metrics[f'{split_name} AUROC (Macro Avg)'] = auroc
    
    return metrics, classification_report(y_true, y_pred)


# --- 4. Compute Metrics for All Splits and Print ---
train_metrics, train_report = compute_metrics(
    X_train, y_train, "Training", xgb_model, is_train=True, feature_time=train_time_feat
)
train_metrics['Training Time (s)'] = train_time_xgb + train_time_feat 

val_metrics, val_report = compute_metrics(
    X_val, y_val, "Validation", xgb_model, feature_time=val_time_feat
)

test_metrics, test_report = compute_metrics(
    X_test, y_test, "Testing", xgb_model, feature_time=test_time_feat
)

# ... (Final printing logic remains the same) ...
print("\n" + "="*80)
print("FINAL HYBRID MODEL METRICS SUMMARY (ViT-Base + EfficientNetV2-S + XGBoost)")
print("="*80)

all_metrics = {**train_metrics, **val_metrics, **test_metrics}
df_metrics = pd.DataFrame(
    list(all_metrics.items()), columns=['Metric', 'Value']
).set_index('Metric')
print(df_metrics)

print("\n" + "-"*30 + " TRAINING CLASSIFICATION REPORT " + "-"*30)
print(train_report)
print("\n" + "-"*30 + " VALIDATION CLASSIFICATION REPORT " + "-"*30)
print(val_report)
print("\n" + "-"*30 + " TESTING CLASSIFICATION REPORT " + "-"*30)
print(test_report)

Loaded training data shape: (18898, 2304)
Starting XGBoost training...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [19:53:05] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



FINAL HYBRID MODEL METRICS SUMMARY (ViT-Base + EfficientNetV2-S + XGBoost)
                                                Value
Metric                                               
Training Accuracy                            0.999894
Training Precision                           0.999893
Training Recall                              0.999889
Training F1 Score                            0.999891
Training TPR (Macro Avg)                     0.999889
Training FPR (Macro Avg)                     0.000013
Training Loss                 N/A (XGBoost Objective)
Training Time (s)                          161.932426
Validation Accuracy                          0.923793
Validation Precision                         0.924184
Validation Recall                            0.921918
Validation F1 Score                          0.922928
Validation TPR (Macro Avg)                   0.921918
Validation FPR (Macro Avg)                   0.009543
Validation Loss                              0.246683
Valida